In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import os
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
# MultiheadAttention Layer
class MultiheadAttention(tf.keras.layers.Layer):
    def __init__(self, num_heads, d_model):
        super(MultiheadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model

        assert d_model % self.num_heads == 0

        self.depth = d_model // self.num_heads

        self.wq = tf.keras.layers.Dense(d_model)
        self.wk = tf.keras.layers.Dense(d_model)
        self.wv = tf.keras.layers.Dense(d_model)

        self.dense = tf.keras.layers.Dense(d_model)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def scaled_dot_product_attention(self, Q, K, V):
        matmul_qk = tf.matmul(Q, K, transpose_b=True)
        dk = tf.cast(self.depth, tf.float32)
        scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)
        attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
        output = tf.matmul(attention_weights, V)
        return output, attention_weights

    def call(self, x):
        batch_size = tf.shape(x)[0]

        q = self.wq(x)
        k = self.wk(x)
        v = self.wv(x)

        q = self.split_heads(q, batch_size)
        k = self.split_heads(k, batch_size)
        v = self.split_heads(v, batch_size)

        scaled_attention, _ = self.scaled_dot_product_attention(q, k, v)
        scaled_attention = tf.transpose(scaled_attention, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(scaled_attention, (batch_size, -1, self.d_model))
        output = self.dense(concat_attention)
        output = tf.reshape(output, (batch_size, self.d_model))

        return output

In [ ]:
# class Generator(tf.keras.Model):
#     def __init__(self, data_dim, h_dim, num_heads):
#         super(Generator, self).__init__()
#         self.fc1 = tf.keras.layers.Dense(h_dim, activation='leaky_relu')
#         self.fc2 = tf.keras.layers.Dense(h_dim, activation='leaky_relu')
#         self.fc3 = tf.keras.layers.Dense(data_dim, activation='sigmoid')
#         self.attention = MultiheadAttention(num_heads, h_dim)
#         self.projection = tf.keras.layers.Dense(data_dim)

#     def call(self, x, m, z, h):
#         attention_output = self.attention(x)
#         attention_output = self.projection(attention_output)
#         x = x + attention_output
#         inputs = x * m + (z * (1 - m))
#         x = self.fc1(inputs)
#         x = self.fc2(x)
#         x = self.fc3(x)
#         return x

# class Generator(tf.keras.Model):
#     def __init__(self, data_dim, h_dim, num_heads):
#         super(Generator, self).__init__()
#         self.fc1 = tf.keras.layers.Dense(h_dim, activation='leaky_relu')
#         self.fc2 = tf.keras.layers.Dense(h_dim, activation='leaky_relu')
#         self.fc3 = tf.keras.layers.Dense(data_dim, activation='sigmoid')
#         self.attention = MultiheadAttention(num_heads, h_dim)
#         self.projection = tf.keras.layers.Dense(data_dim)

#     def call(self, x, m, z, h):
#         attention_output = self.attention(m)
#         attention_output = self.projection(attention_output)
#         m = m + attention_output
#         inputs = x * m + (z * (1 - m))
#         x = self.fc1(inputs)
#         x = self.fc2(x)
#         x = self.fc3(x)
#         return x

class Generator(tf.keras.Model):
    def __init__(self, data_dim, h_dim, num_heads):
        super(Generator, self).__init__()
        self.fc1 = tf.keras.layers.Dense(h_dim, activation='leaky_relu')
        self.fc2 = tf.keras.layers.Dense(h_dim, activation='leaky_relu')
        self.fc3 = tf.keras.layers.Dense(data_dim, activation='sigmoid')
        self.attention = MultiheadAttention(num_heads, h_dim)
        self.projection = tf.keras.layers.Dense(data_dim)

    def call(self, x, m, z, h):
        attention_output = self.attention(z)
        attention_output = self.projection(attention_output)
        z = z + attention_output
        inputs = x * m + (z * (1 - m))
        x = self.fc1(inputs)
        x = self.fc2(x)
        x = self.fc3(x)
        return x

# class Generator(tf.keras.Model):
#     def __init__(self, data_dim, h_dim, num_heads):
#         super(Generator, self).__init__()
#         self.fc1 = tf.keras.layers.Dense(h_dim, activation='leaky_relu')
#         self.fc2 = tf.keras.layers.Dense(h_dim, activation='leaky_relu')
#         self.fc3 = tf.keras.layers.Dense(data_dim, activation='sigmoid')
#         self.attention = MultiheadAttention(num_heads, h_dim)
#         self.projection = tf.keras.layers.Dense(data_dim)

#     def call(self, x, m, z, h):
#         inputs = x * m + (z * (1 - m))
#         attention_output = self.attention(inputs)
#         attention_output = self.projection(attention_output)
#         x = x + attention_output
#         x = self.fc1(inputs)
#         x = self.fc2(x)
#         x = self.fc3(x)
#         return x

In [ ]:
class Critic(tf.keras.Model):
    def __init__(self, data_dim, h_dim, num_heads):
        super(Critic, self).__init__()
        self.fc1 = tf.keras.layers.Dense(h_dim, activation='leaky_relu')
        self.fc2 = tf.keras.layers.Dense(h_dim, activation='leaky_relu')
        self.fc3 = tf.keras.layers.Dense(data_dim, activation='sigmoid')
        self.attention = MultiheadAttention(num_heads, h_dim)
        self.projection = tf.keras.layers.Dense(data_dim)

    def call(self, x, h):
        inputs = tf.concat([x, h], axis=1)
        x = self.fc1(inputs)
        x = self.fc2(x)
        x = self.fc3(x)
        return x

In [ ]:
def process_dataset(file_path, output_path, iterations=10000):
    data = pd.read_csv(file_path)
    original_data_item_area_year = data.copy()
    data = data.drop(columns=['Item', 'Area', 'Year'])
    original_data = data.copy()

    def binary_sampler (p, rows, cols, seed=43):
      np.random.seed(seed)

      unif_random_matrix = np.random.uniform(0., 1., size = [rows, cols])
      binary_random_matrix = 1*(unif_random_matrix < p)
      return binary_random_matrix

    # Gradient penalty (penalizes deviations from the Lipschitz constraint)
    def gradient_penalty(real, fake, mask):
        epsilon = tf.random.uniform([real.shape[0], 1], 0.0, 1.0)
        interpolated = epsilon * real + (1 - epsilon) * fake
        with tf.GradientTape() as gp_tape:
            gp_tape.watch(interpolated)
            inter_output = critic(interpolated, mask)
        grads = gp_tape.gradient(inter_output, [interpolated])[0]
        grad_penalty = tf.reduce_mean((tf.norm(grads, axis=1) - 1.0) ** 2)
        return grad_penalty

    @tf.function
    def train_step(X_mb, M_mb, Z_mb, H_mb):
        with tf.GradientTape() as crit_tape, tf.GradientTape() as gen_tape:
            G_sample = generator(X_mb, M_mb, Z_mb, H_mb)
            Hat_X = X_mb * M_mb + G_sample * (1 - M_mb)

            D_real = critic(X_mb, H_mb)
            D_fake = critic(Hat_X, H_mb)

            gp = gradient_penalty(X_mb, Hat_X, H_mb)
            gp = lambda_gp * gp

            D_loss_real = tf.reduce_mean(M_mb * tf.maximum(tf.math.log(tf.maximum(D_real, 1e-8)), 1e-8) + (1 - M_mb) * tf.maximum(tf.math.log(1.0 - tf.maximum(D_real, 1e-8)), 1e-8))
            D_loss_fake = tf.reduce_mean(M_mb * tf.maximum(tf.math.log(tf.maximum(D_fake, 1e-8)), 1e-8) + (1 - M_mb) * tf.maximum(tf.math.log(1.0 - tf.maximum(D_fake, 1e-8)), 1e-8))
            D_loss = D_loss_fake - D_loss_real + gp

            G_loss = tf.reduce_mean((1 - M_mb) * tf.maximum(tf.math.log(tf.maximum(D_fake, 1e-8)), 1e-8))
            M_float = tf.cast(M_mb, tf.float32)
            MSE_loss = tf.reduce_mean(tf.square((M_float * X_mb - M_float * G_sample)**2)) / tf.reduce_mean(M_float)
            G_loss = G_loss + alpha * MSE_loss

        crit_gradients = crit_tape.gradient(D_loss, critic.trainable_variables)
        gen_gradients = gen_tape.gradient(G_loss, generator.trainable_variables)

        crit_optimizer.apply_gradients(zip(crit_gradients, critic.trainable_variables))
        gen_optimizer.apply_gradients(zip(gen_gradients, generator.trainable_variables))

        return D_loss, G_loss

    miss_rates = [0.2]
    hint_rates = [0.9]

    for miss_rate in miss_rates:
        for hint_rate in hint_rates:
            print(f'Miss rate: {miss_rate}, hint rate: {hint_rate}')

            missing_mask = binary_sampler(1 - miss_rate, original_data.shape[0], original_data.shape[1])

            original_missing_value = np.where(np.isnan(original_data) == True, 1, 0)
            missing_mask_wo_missing_value = missing_mask + original_missing_value

            data_with_nan = original_data.copy()
            data_with_nan[missing_mask_wo_missing_value == 0] = np.nan

            scaler = MinMaxScaler()
            data_scaled = scaler.fit_transform(data_with_nan)
            data_scaled = np.nan_to_num(data_scaled, 0)

            # MCAR missingness
            np.random.seed(42)

            # Define the GAIN model parameters
            num_heads = 7
            data_dim = data_scaled.shape[1]
            h_dim = 21
            batch_size = 64
            alpha = 1
            iterations = 10000
            lambda_gp = 5

            # Instantiate the models
            generator = Generator(data_dim, h_dim, num_heads)
            critic = Critic(data_dim, h_dim, num_heads)

            # Optimizers
            gen_optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
            crit_optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

            for it in range(iterations):
                batch_idx = np.random.choice(range(data_scaled.shape[0]), size=batch_size, replace=False)
                X_mb = data_scaled[batch_idx, :].astype(np.float32)
                M_mb = 1 - np.isnan(data_with_nan.to_numpy())[batch_idx, :].astype(np.float32)
                Z_mb = np.random.uniform(0, 0.01, size=[batch_size, data_dim]).astype(np.float32)
                H_mb_temp = np.random.binomial(1, hint_rate, size=[batch_size, data_dim]).astype(np.float32)
                H_mb = M_mb * H_mb_temp

                D_loss_curr, G_loss_curr = train_step(X_mb, M_mb, Z_mb, H_mb)

                # if it % 100 == 0:
                #     print(f'Iter: {it}; D_loss: {D_loss_curr:.4}; G_loss: {G_loss_curr:.4}')

            Z_mb = np.random.uniform(0, 0.01, size=[data_scaled.shape[0], data_dim]).astype(np.float32)
            M_mb = 1 - np.isnan(data_scaled).astype(np.float32)
            X_mb = data_scaled.astype(np.float32)

            imputed_data = generator(X_mb, M_mb, Z_mb, H_mb).numpy()

            imputed_data = scaler.inverse_transform(imputed_data)

            imputed_data_df = pd.DataFrame(imputed_data, columns=data.columns)

            output_file = f"{output_path}_miss_{miss_rate}_hint_{hint_rate}"
            imputed_data_with_original_columns = imputed_data_df.copy()
            imputed_data_with_original_columns['Item'] = original_data_item_area_year['Item'].values
            imputed_data_with_original_columns['Area'] = original_data_item_area_year['Area'].values
            imputed_data_with_original_columns['Year'] = original_data_item_area_year['Year'].values

            cols = ['Item', 'Area', 'Year'] + [col for col in imputed_data_with_original_columns.columns if col not in ['Item', 'Area', 'Year']]
            imputed_data_with_original_columns = imputed_data_with_original_columns[cols]
            imputed_data_with_original_columns.to_csv(output_file, index=False)
            print(f"Imputed data saved to {output_file}")

            mcar_mask = np.where(missing_mask_wo_missing_value > 0, 1, 0)
            ori_data = original_data.fillna(0).to_numpy()
            imputed_data_mcar = imputed_data[mcar_mask]

            nominator = np.sum(((1 - mcar_mask) * ori_data - (1 - mcar_mask) * imputed_data) ** 2)
            denominator = np.sum(1 - mcar_mask)

            rmse = np.sqrt(nominator / float(denominator))
            print(f'RMSE: {rmse:.4f}')
            mse = nominator / float(denominator)
            print(f'MSE: {mse:.4f}')

            nominator = np.sum(np.abs((1 - mcar_mask) * ori_data - (1 - mcar_mask) * imputed_data))
            denominator = np.sum(1 - mcar_mask)
            mae = nominator / float(denominator)
            print(f'MAE: {mae:.4f}')

In [ ]:
datasets = {
    "Rice": "/content/Rice.csv",
    "Soya": "/content/Soya.csv",
    "Corn": "/content/Corn.csv"
}

In [ ]:
for name, file_path in datasets.items():
    output_path = f'/content/{name}_imputed_GAIN_Wassertein_Attention(z)_all'
    print(f"Processing {name} dataset...")
    process_dataset(file_path, output_path)

Processing Rice dataset...
Miss rate: 0.2, hint rate: 0.9


/usr/local/lib/python3.10/dist-packages/keras/src/layers/layer.py:391: UserWarning: `build()` was called on layer 'critic_15', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Imputed data saved to /content/Rice_imputed_GAIN_Wassertein(z)_all_miss_0.2_hint_0.9
RMSE: 3949569.4451
MSE: 15599098801663.9004
MAE: 1121639.8444
Processing Soya dataset...
Miss rate: 0.2, hint rate: 0.9


/usr/local/lib/python3.10/dist-packages/keras/src/layers/layer.py:391: UserWarning: `build()` was called on layer 'critic_16', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Imputed data saved to /content/Soya_imputed_GAIN_Wassertein(z)_all_miss_0.2_hint_0.9
RMSE: 204091.2215
MSE: 41653226679.2908
MAE: 89868.4043
Processing Corn dataset...
Miss rate: 0.2, hint rate: 0.9


/usr/local/lib/python3.10/dist-packages/keras/src/layers/layer.py:391: UserWarning: `build()` was called on layer 'critic_17', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Imputed data saved to /content/Corn_imputed_GAIN_Wassertein(z)_all_miss_0.2_hint_0.9
RMSE: 822434.3257
MSE: 676398220150.0306
MAE: 280396.2153
